# Module 05: Flight Software & Attitude Control

This is the core FSW (Flight Software) module. We wire together:
1. **Attitude reference** — where do we want to point?
2. **Attitude error** — where are we vs where we want to be?
3. **Control law** — what torques to command?
4. **Actuator allocation** — how to distribute torque across reaction wheels?

### Learning Objectives
- Configure `inertial3D` pointing guidance
- Use `attTrackingError` to compute MRP error
- Implement `MRP_Feedback` PD control law
- Use `rwMotorTorque` for RW torque allocation
- Close the loop between dynamics and FSW

---

## Architecture: The FSW Pipeline

```
┌─────────────────────────────────────────────────────────────────┐
│  DYNAMICS (1 Hz)          │  FSW (2 Hz)                        │
│                           │                                    │
│  scObject ──────────────────► attTrackingError                │
│  (scStateOutMsg)          │      │                             │
│       ▲                   │  inertial3D ──► attTrackingError  │
│       │ (torque)          │      (ref)          │ (error)      │
│  rwStateEffector ◄─────────── rwMotorTorque ◄── MRP_Feedback  │
│  (rwMotorTorqueInMsg)     │      (alloc)        (control)      │
└─────────────────────────────────────────────────────────────────┘
```

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import os
os.makedirs('plots', exist_ok=True)

from Basilisk.utilities import SimulationBaseClass, macros, orbitalMotion, unitTestSupport
from Basilisk.utilities import RigidBodyKinematics as rbk
from Basilisk.simulation import spacecraft, gravityEffector, reactionWheelStateEffector
from Basilisk.fswAlgorithms import inertial3D         # guidance: inertial pointing
from Basilisk.fswAlgorithms import attTrackingError   # error computation
from Basilisk.fswAlgorithms import MRP_Feedback       # PD control law
from Basilisk.fswAlgorithms import rwMotorTorque      # RW torque allocation
from Basilisk.architecture  import bskLogging

bskLogging.setDefaultLogLevel(bskLogging.BSK_WARNING)
print("Imports OK.")

---

## Step 1 — Simulation Skeleton with Two Processes

In [ ]:
scSim = SimulationBaseClass.SimBaseClass()

# ── Two processes: dynamics runs before FSW ────────────────────────────────────
dynProcess = scSim.CreateNewProcess("DynamicsProcess", priority=10)
fswProcess = scSim.CreateNewProcess("FswProcess",      priority=5)

dynTimeStep = macros.sec2nano(0.5)   # 2 Hz dynamics
fswTimeStep = macros.sec2nano(1.0)   # 1 Hz FSW

dynTask = scSim.CreateNewTask("DynamicsTask", dynTimeStep)
fswTask = scSim.CreateNewTask("FswTask",      fswTimeStep)

dynProcess.addTask(dynTask)
fswProcess.addTask(fswTask)

print("Simulation skeleton created.")

---

## Step 2 — Spacecraft with Reaction Wheels

In [ ]:
mu_earth = 3.986004418e14
R_earth  = 6371e3

# ── Spacecraft ────────────────────────────────────────────────────────────────
scObject = spacecraft.Spacecraft()
scObject.ModelTag = "BSKsat"

Ixx, Iyy, Izz = 0.025, 0.050, 0.065   # kg*m^2 for a small sat
I3x3 = np.diag([Ixx, Iyy, Izz])
scObject.hub.mHub            = 10.0
scObject.hub.IHubPntBc_B     = unitTestSupport.np2EigenMatrix3d(I3x3)
scObject.hub.r_BcB_B         = [[0.0], [0.0], [0.0]]

# ── Initial orbit ─────────────────────────────────────────────────────────────
oe = orbitalMotion.ClassicElements()
oe.a = R_earth + 500e3
oe.e = 0.0
oe.i = 28.5 * macros.D2R
oe.Omega = oe.omega = oe.f = 0.0
r0, v0 = orbitalMotion.elem2rv(mu_earth, oe)

scObject.hub.r_CN_NInit = r0.tolist()
scObject.hub.v_CN_NInit = v0.tolist()

# ── Initial attitude: large pointing error (should be corrected by control) ───
sigma_init = np.array([0.3, 0.1, -0.2])   # large initial error
omega_init = np.array([0.01, -0.01, 0.005])  # small initial tumble (rad/s)
scObject.hub.sigma_BNInit    = [[sigma_init[0]], [sigma_init[1]], [sigma_init[2]]]
scObject.hub.omega_BN_BInit  = [[omega_init[0]], [omega_init[1]], [omega_init[2]]]

# ── Gravity ───────────────────────────────────────────────────────────────────
gravFactory = gravityEffector.GravBodyDataVector()
earth = gravFactory.createEarth()
earth.isCentralBody = True
scObject.gravField.gravBodies = gravFactory

scSim.AddModelToTask("DynamicsTask", scObject)

# ── Reaction Wheels (3-axis orthogonal) ───────────────────────────────────────
rwFactory = reactionWheelStateEffector.rwFactory()
varRWModel = reactionWheelStateEffector.BalancedWheels

rw_axes = [[1,0,0], [0,1,0], [0,0,1]]
for axis in rw_axes:
    rwFactory.create('Honeywell_HR16',
                     gsHat_B=axis,
                     maxMomentum=50.0,
                     Omega=0.0,
                     RWModel=varRWModel)

rwStateEffector = reactionWheelStateEffector.ReactionWheelStateEffector()
rwStateEffector.ModelTag = "RWArray"
rwFactory.addToSpacecraft(scObject.ModelTag, rwStateEffector, scObject)
scSim.AddModelToTask("DynamicsTask", rwStateEffector)

print(f"Spacecraft + {rwFactory.getNumOfRW()} RWs configured.")

---

## Step 3 — FSW: Guidance Module (`inertial3D`)

`inertial3D` commands the spacecraft to hold a fixed inertial orientation. It outputs an `attRefMsg` describing the desired attitude.

In [ ]:
# ── Inertial pointing: command body to align with inertial frame ───────────────
inertial3DObj = inertial3D.inertial3D()
inertial3DObj.ModelTag = "inertial3D"

# sigma_R0N: desired reference attitude (body aligned with inertial N-frame)
inertial3DObj.sigma_R0N = [0.0, 0.0, 0.0]  # no rotation from inertial

scSim.AddModelToTask("FswTask", inertial3DObj)

print("inertial3D guidance configured.")

---

## Step 4 — FSW: Attitude Tracking Error (`attTrackingError`)

This module computes the error between the current attitude (`navAttInMsg`) and the desired reference (`attRefInMsg`), outputting `attGuidOutMsg` with σ_BR and ω_BR.

In [ ]:
attErrObj = attTrackingError.attTrackingError()
attErrObj.ModelTag = "attTrackingError"

# Connect navigation (current attitude from spacecraft) to the error module
attErrObj.attNavInMsg.subscribeTo(scObject.attOutMsg)    # from dynamics
attErrObj.attRefInMsg.subscribeTo(inertial3DObj.attRefOutMsg)  # from guidance

scSim.AddModelToTask("FswTask", attErrObj)

print("attTrackingError configured.")

---

## Step 5 — FSW: MRP Feedback Control Law

The `MRP_Feedback` module implements a **PD control law** on MRP:

$$\mathbf{u} = -K\boldsymbol{\sigma}_{B/R} - P\boldsymbol{\omega}_{B/R}$$

where:
- **K** = proportional (attitude) gain
- **P** = derivative (rate) gain  
- σ_B/R = attitude error MRP
- ω_B/R = angular velocity error

In [ ]:
# ── MRP Feedback control gains ────────────────────────────────────────────────
# Tuning guidelines:
#   K ~ 2*zeta*wn * sum(I) — stiffness (attitude)
#   P ~ wn^2 * sum(I)      — damping (rate)
#
# Choose natural frequency wn = 0.15 rad/s, damping ratio zeta = 0.7
wn   = 0.15      # natural frequency [rad/s]
zeta = 0.7       # damping ratio
I_total = Ixx + Iyy + Izz

K = 2.0 * zeta * wn * I_total
P = wn**2 * I_total

print(f"Control gains: K = {K:.5f}, P = {P:.5f}")

mrpControlObj = MRP_Feedback.MRP_Feedback()
mrpControlObj.ModelTag  = "MRP_Feedback"
mrpControlObj.K         = K
mrpControlObj.P         = P
mrpControlObj.integralLimit = 2.0 / mrpControlObj.Ki  # disable integral term
mrpControlObj.Ki        = -1.0   # disable integral gain

# Connect messages
mrpControlObj.guidInMsg.subscribeTo(attErrObj.attGuidOutMsg)
mrpControlObj.vehConfigInMsg.subscribeTo(scObject.vehicleConfigOutMsg)

# Subscribe RW speeds into the controller (feeds back RW angular momentum)
mrpControlObj.rwParamsInMsg.subscribeTo(rwFactory.rwParamsMsgs[0])  # config
mrpControlObj.rwSpeedsInMsg.subscribeTo(rwStateEffector.rwSpeedOutMsg)

scSim.AddModelToTask("FswTask", mrpControlObj)

print("MRP_Feedback control law configured.")

---

## Step 6 — FSW: RW Torque Allocation (`rwMotorTorque`)

In [ ]:
rwMotorTorqueObj = rwMotorTorque.rwMotorTorque()
rwMotorTorqueObj.ModelTag = "rwMotorTorque"

# controlAxes_B: which body axes should be controlled
rwMotorTorqueObj.controlAxes_B = [1, 0, 0,
                                   0, 1, 0,
                                   0, 0, 1]

# Connect messages
rwMotorTorqueObj.vehControlInMsg.subscribeTo(mrpControlObj.cmdTorqueOutMsg)
rwMotorTorqueObj.rwParamsInMsg.subscribeTo(rwFactory.rwParamsMsgs[0])

# The output motor torque message connects to the RW state effector
rwStateEffector.rwMotorCmdInMsg.subscribeTo(rwMotorTorqueObj.rwMotorTorqueOutMsg)

scSim.AddModelToTask("FswTask", rwMotorTorqueObj)

print("rwMotorTorque allocation configured.")

---

## Step 7 — Data Recording

In [ ]:
# ── Record attitude state ─────────────────────────────────────────────────────
attStateRec = scObject.scStateOutMsg.recorder()
scSim.AddModelToTask("DynamicsTask", attStateRec)

# ── Record attitude error ─────────────────────────────────────────────────────
attErrRec = attErrObj.attGuidOutMsg.recorder()
scSim.AddModelToTask("FswTask", attErrRec)

# ── Record RW motor torques and speeds ────────────────────────────────────────
rwTorqueRec = rwMotorTorqueObj.rwMotorTorqueOutMsg.recorder()
rwSpeedRec  = rwStateEffector.rwSpeedOutMsg.recorder()
scSim.AddModelToTask("FswTask",      rwTorqueRec)
scSim.AddModelToTask("DynamicsTask", rwSpeedRec)

print("Recorders attached.")

---

## Step 8 — Initialize & Run

In [ ]:
scSim.InitializeSimulation()
scSim.ConfigureStopTime(macros.sec2nano(300.0))   # 5 minutes
scSim.ExecuteSimulation()

print(f"Simulation complete. t_final = {scSim.TotalSim.CurrentNanos * macros.NANO2SEC:.0f} s")

---

## Step 9 — Analyze Results

In [ ]:
t_dyn  = attStateRec.times() * macros.NANO2SEC
sigma  = attStateRec.sigma_BN     # body attitude MRP
omega  = attStateRec.omega_BN_B   # body angular velocity

t_fsw   = attErrRec.times() * macros.NANO2SEC
sigma_e = attErrRec.sigma_BR      # attitude error MRP
omega_e = attErrRec.omega_BR_B    # angular velocity error

rwOmega  = rwSpeedRec.wheelSpeeds  # RW speeds [N x 3]
rwTorque = rwTorqueRec.motorTorque # RW motor torques [N x 3]

# Attitude error magnitude in degrees
err_deg = np.degrees(4 * np.arctan(np.linalg.norm(sigma_e, axis=1)))

print(f"Initial attitude error: {err_deg[0]:.2f} deg")
print(f"Final   attitude error: {err_deg[-1]:.4f} deg")
print(f"Settling time (< 1 deg): ", end="")
settled = np.where(err_deg < 1.0)[0]
if len(settled) > 0:
    print(f"{t_fsw[settled[0]]:.1f} s")
else:
    print("Not settled within simulation time")

In [ ]:
# ── Plot attitude error convergence ──────────────────────────────────────────
fig, axes = plt.subplots(2, 1, figsize=(12, 8), sharex=True)

axes[0].plot(t_fsw, err_deg, color='crimson', linewidth=2)
axes[0].axhline(1.0, color='gray', linestyle='--', label='1 deg threshold')
axes[0].axhline(0.1, color='lightgray', linestyle=':', label='0.1 deg threshold')
axes[0].set_ylabel('Attitude Error (deg)')
axes[0].set_yscale('log')
axes[0].set_title('MRP Feedback Attitude Control — Error Convergence')
axes[0].legend()
axes[0].grid(True, alpha=0.3)

axes[1].plot(t_fsw, np.linalg.norm(omega_e, axis=1) * macros.R2D,
             color='darkorange', linewidth=2)
axes[1].set_xlabel('Time (s)')
axes[1].set_ylabel('Rate Error (deg/s)')
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('plots/05_attitude_error_convergence.png', dpi=100, bbox_inches='tight')
plt.show()

# ── Plot RW motor torques ─────────────────────────────────────────────────────
fig, axes = plt.subplots(3, 1, figsize=(12, 9), sharex=True)
rw_labels = ['RW-X', 'RW-Y', 'RW-Z']
colors = ['tab:blue', 'tab:orange', 'tab:green']

for i in range(3):
    axes[i].plot(t_fsw, rwTorque[:, i] * 1e3, color=colors[i])
    axes[i].set_ylabel(f'{rw_labels[i]} Torque (mN·m)')
    axes[i].grid(True, alpha=0.3)

axes[0].set_title('Reaction Wheel Motor Torques')
axes[2].set_xlabel('Time (s)')
plt.tight_layout()
plt.savefig('plots/05_rw_torques.png', dpi=100, bbox_inches='tight')
plt.show()

# ── Plot RW speeds ────────────────────────────────────────────────────────────
fig, ax = plt.subplots(figsize=(12, 5))
for i in range(3):
    ax.plot(t_dyn, rwOmega[:, i] / macros.rpm2rads,
            label=rw_labels[i], color=colors[i])
ax.set_xlabel('Time (s)')
ax.set_ylabel('RW Speed (RPM)')
ax.set_title('Reaction Wheel Speeds During Attitude Maneuver')
ax.legend()
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig('plots/05_rw_speeds_controlled.png', dpi=100, bbox_inches='tight')
plt.show()

---

## Exercises

1. **Tune the controller**: Increase `wn` for faster settling. What happens to the torque peaks?
2. **Larger initial error**: Set `sigma_init = [0.6, 0.2, -0.5]`. Does the controller still converge?
3. **Different guidance**: Replace `inertial3D` with `hillPoint` for nadir-pointing.
4. **Saturation**: Add maximum torque limits to the RW motor torque. What effect does saturation have?

---

## Summary — FSW Pipeline

| Module | Class | Output Message | Purpose |
|---|---|---|---|
| Guidance | `inertial3D` | `attRefOutMsg` | Desired σ_RN, ω_RN |
| Error | `attTrackingError` | `attGuidOutMsg` | σ_BR, ω_BR errors |
| Control | `MRP_Feedback` | `cmdTorqueOutMsg` | Required control torque |
| Allocation | `rwMotorTorque` | `rwMotorTorqueOutMsg` | Per-wheel torque commands |

**Next: [06 - Sensors & Actuators](06_sensors_actuators.ipynb)**